# Разговор с агентами в кластере (части 6–7)

Агенты запущены в Kubernetes, а мы — на ноутбуке. Снаружи виден **только риск-менеджер**: через
Ingress по адресу `http://risk.agents.test`. Аналитик и MCP-сервер спрятаны внутри кластера —
к ним ходит сам риск-менеджер.

**Перед запуском:** применены части 5–6 (`kubectl apply -f part_5_bank_db/ …`, `kubectl apply -f part_6_agents/`)
и прописаны домены (`make hosts`). Держите рядом терминал с логами:

```bash
kubectl logs -f -l app=risk-manager --prefix
```

## 1. Карточка агента — через Ingress

In [ ]:
import json

import httpx

RISK_URL = "http://risk.agents.test"

card = httpx.get(f"{RISK_URL}/.well-known/agent-card.json").json()
print(json.dumps({k: card[k] for k in ("name", "description", "supportedInterfaces", "skills")},
                 indent=2, ensure_ascii=False))

## 2. Вопрос риск-менеджеру

Клиент — тот же `ask.py`, что лежит в образе агентов (им пользуются и CronJob, и `kubectl run`).
Он возвращает ответ, **какой под ответил** и номер разговора `context_id`.

In [ ]:
import sys

sys.path.insert(0, "images/agents")
from ask import ask                       # images/agents/ask.py

reply = await ask(RISK_URL, "Можно ли выдать счёту 2378 кредит 150 000 CZK на 24 месяца?")
print(reply["answer"])
print("\n— под:", reply["pod"], "| разговор:", reply["context_id"])

## 3. Уточнение в том же разговоре

Передаём тот же `context_id` — риск-менеджер помнит, о каком счёте речь.

In [ ]:
follow_up = await ask(RISK_URL, "А если сумма 300 000 на тот же срок? Коротко.", reply["context_id"])
print(follow_up["answer"])
print("\n— под:", follow_up["pod"])

## 4. Память и реплики (часть 7)

Функция задаёт вопрос, а потом 4 раза спрашивает в **том же разговоре** номер счёта и печатает,
какой под ответил. Запускайте её в трёх состояниях кластера:

| состояние | команда в терминале | что ожидаем |
|---|---|---|
| A. 1 реплика, память в RAM | (как после части 6) | все ответы — `2378`, под один |
| B. 2 реплики, память в RAM | `kubectl scale deploy/risk-manager --replicas=2` | у «чужого» пода — `НЕ ЗНАЮ` |
| C. 2 реплики, память в Postgres | `kubectl apply -f part_7_scale_state/risk-manager-shared-memory.yaml` | все ответы — `2378`, поды разные |

In [ ]:
async def memory_test() -> None:
    first = await ask(RISK_URL, "Можно ли выдать счёту 2378 кредит 150 000 CZK на 24 месяца? "
                                "Ответь одним словом: ОДОБРИТЬ или ОТКАЗАТЬ.")
    print(f"вопрос   → под {first['pod']}: {first['answer'][:40]!r}")
    for i in range(4):
        r = await ask(RISK_URL, "Напомни номер счёта из нашего разговора — одним числом. "
                                "Если в разговоре его не было, ответь: НЕ ЗНАЮ.", first["context_id"])
        print(f"уточнение {i + 1} → под {r['pod']}: {r['answer'][:40]!r}")


await memory_test()

## 5. Несколько клиентов одновременно

Три решения параллельно. Если реплик риск-менеджера несколько, Ingress раскидает запросы по ним, а
риск-менеджеры — свои вопросы по аналитикам (смотрите `kubectl logs -f -l app=analyst --prefix`).

In [ ]:
import asyncio
import time

questions = ["Можно ли выдать счёту 2378 кредит 150 000 CZK на 24 месяца?",
             "Можно ли выдать счёту 1583 кредит 300 000 CZK на 36 месяцев?",
             "Можно ли выдать счёту 1 кредит 100 000 CZK на 24 месяца?"]

started = time.time()
replies = await asyncio.gather(*(ask(RISK_URL, q) for q in questions))
for q, r in zip(questions, replies):
    print(f"{q[22:60]:<40} → {r['answer'].split(chr(10))[0][:40]:<40} (под {r['pod']})")
print(f"\nвсего {time.time() - started:.0f} с на три решения")

## Итоги

- снаружи кластера доступен только тот агент, у которого есть Ingress; остальные — внутренние сервисы;
- агенты без состояния (аналитик, MCP) масштабируются одной командой `kubectl scale`;
- агенту с памятью нужно общее хранилище — иначе на нескольких репликах разговоры «теряются».